## import libraries

In [1]:
import numpy as np

# Optimization of class scheduling problem with ACO

## Problem setup

In [23]:
classes = [f'C{i+1}' for i in range(12)]
professors = {
    'C1': 'P1', 'C2': 'P1',
    'C3': 'P2', 'C4': 'P2',
    'C5': 'P3', 'C6': 'P3',
    'C7': 'P4', 'C8': 'P4',
    'C9': 'P5', 'C10': 'P5',
    'C11': 'P6', 'C12': 'P6'
}
timeslots = [f'T{i+1}' for i in range(6)]
n_classes = len(classes)
n_times = len(timeslots)
max_classes_per_time = 3


## Constraints

In [24]:
# Professors availability (each can only teach in some time slots)
availability = {
    'P1': [0, 1, 2],
    'P2': [1, 2, 3],
    'P3': [2, 3, 4],
    'P4': [0, 4, 5],
    'P5': [0, 2, 5],
    'P6': [1, 3, 5]
}

# Class pairs that cannot overlap (students overlap)
cannot_overlap = [('C1', 'C3'), ('C4', 'C6'), ('C2', 'C8'), ('C5', 'C10'), ('C7', 'C11')]


## Initialize pheromones and heuristic

In [25]:
#TODO: initialze these parameters

pheromone = np.ones((n_classes,n_times),dtype=float)

heuristic = np.zeros((n_classes,n_times))
for index,class_ in enumerate(classes):
    professor=professors[class_]
    for time_slot in availability[professor]:
        heuristic[index,time_slot]=1


## Ant builds a solution

In [ ]:
#TODO: complete construct_solution function
def construct_solution(pheromone, heuristic, alpha=1, beta=2):
    schedule = np.zeros(n_classes,dtype=int)
    all_classes=list(range(n_classes))

    while all_classes!=[]:
        class_=np.random.choice(all_classes)
        probabilities=[]
        for i in range(n_times):
            probabilities.append((pheromone[class_,i]**alpha)*(heuristic[class_,i]**beta))
        
        probabilities=np.array(probabilities)
        probabilities/=probabilities.sum()

        schedule[class_]=np.random.choice(n_times,p=probabilities)
        all_classes.remove(class_)


    return schedule


## Evaluate constraints

In [27]:
# TODO: complete evaluate_conflicts
def evaluate_conflicts(schedule):
    conflicts = 0
    time_table = {t: [] for t in range(n_times)}

    for i, time in enumerate(schedule):
        cls = classes[i]
        prof = professors[cls]
        time_table[time].append((cls, prof))

    #TODO: Check that the same professor is not in multiple classes
    for i in range(0,n_classes,2):
        if schedule[i]==schedule[i+1]:
            conflicts+=20


    #TODO: Check the number of classes at any one time is not more than 3
    for time,plan in time_table.items():
        if len(plan)>3:
            conflicts+=10*len(plan)-3

    #TODO: Check Professor availability
    for time,plan in time_table.items():
        for i in range(len(plan)):
            if time not in availability[plan[i][1]]:
                conflicts+=10


    #TODO: Check Forbidden class overlaps
    for class1,class2 in cannot_overlap:
        i=classes.index(class1)
        j=classes.index(class2)
        if schedule[i]==schedule[j]:
            conflicts+=30

    return conflicts


## Pheromone update

In [28]:
#TODO: Pheromone update
def update_pheromone(pheromone, solutions, scores, rho=0.1, Q=100):
    pheromone=pheromone*(1-rho)

    for solution , score in zip(solutions,scores):
        x=Q/(1+score)
        for class_,time_slot in enumerate(solution):
            pheromone[class_,time_slot]+=x


## ACO main loop

In [29]:
n_ants = 10
n_iter = 30
best_sol = None
best_score = float('inf')
history = []

for it in range(n_iter):

    #TODO: complete ACO main loop
    solutions = [construct_solution(pheromone, heuristic) for _ in range(n_ants)]
    scores=[evaluate_conflicts(solution) for solution in solutions]
    update_pheromone(pheromone,solutions,scores)
    best=np.min(scores)
    if best<best_score:
        best_score=best
        best_idx = np.argmin(scores)
        best_sol=solutions[best_idx]

    history.append(best_score)


## show result


In [31]:
print("Best schedule with minimum conflicts:")
for i, time in enumerate(best_sol):
    print(f"Class {classes[i]} → Time {timeslots[time]}")
print("Total conflicts:", best_score)


Best schedule with minimum conflicts:
Class C1 → Time T1
Class C2 → Time T3
Class C3 → Time T2
Class C4 → Time T3
Class C5 → Time T3
Class C6 → Time T4
Class C7 → Time T6
Class C8 → Time T1
Class C9 → Time T1
Class C10 → Time T6
Class C11 → Time T2
Class C12 → Time T4
Total conflicts: 0
